In [ ]:
import CAST
import scanpy as sc
import torch, os
import numpy as np
from os.path import join as pj
import warnings
warnings.filterwarnings("ignore")
import pandas as pd 
import anndata 
import os.path as osp
os.environ['CUDA_VISIBLE_DEVICES']='0' 
device='cuda' if torch.cuda.is_available() else 'cpu'
print(device)

In [ ]:
slice_ids = ['151507', '151508',  '151509', '151510']

def make_unique(s):
    if s in counts:
        counts[s] += 1
    else:
        counts[s] = 0
    return s if counts[s] == 0 else f"{s}_{counts[s]}"  
slices=[]
for j,section_ids in enumerate(slice_ids):
    slice1 = load_DLPFC(section_id=section_ids)
    slice1.layers["counts"] = slice1.X.copy()
    slice1.obs['Z']=j
    slice1.obs['slice_id']=section_ids
    counts = {}
    slice1.var.index = slice1.var.index.map(make_unique)
    slice1.obs_names_make_unique()
    slices.append(slice1)

In [ ]:
from CAST.utils import detect_highly_variable_genes
from CAST.visualize import plot_mid
from CAST.utils import extract_coords_exp
from CAST.models.model_GCNII import Args
from CAST import CAST_MARK
from CAST.visualize import kmeans_plot_multiple
from CAST.CAST_Stack import reg_params
from CAST import CAST_STACK
import pickle as pkl
import time 
from sklearn.metrics import adjusted_rand_score
from sklearn.metrics.pairwise import euclidean_distances
from scipy import stats
from scipy.sparse import issparse

DLPFC data

In [ ]:
path_ = '/home/data3/yuling/xenium_region2_5599/CAST-main/res/DLPFC_4/151507_151510'
seeds_to_run=[2024,1,2,3,4,10,11,12,13,14]
run_time_iter = 0

for j in seeds_to_run:
    slices_aligned = []
    ds = []
    fold_path = osp.join(path_, str(j))
    if not os.path.exists(fold_path):
        os.makedirs(fold_path)
    slices_aligned.append(slices[0])
    time_3d = []
    for i in range(1,len(slice_ids)):
        start_time = time.time()
        slice1 = slices[i]
        slice2 = slices[i-1]
        sdata = slice1.concatenate(slice2) 

# combine dataset 
        batch_key = 'batch'
        sample_list= [str(slice_ids[i]),str(slice_ids[i-1])] # [Query, Reference]
        batch_rename = {'0' : sample_list[0],'1' : sample_list[1]}
        sdata.obs.replace({batch_key:batch_rename},inplace=True)

# Filter for highly variable genes
        sdata.var['highly_variable'] = detect_highly_variable_genes(sdata,batch_key='batch',n_top_genes=4000,count_layer='.X')
        sdata = sdata[:,sdata.var['highly_variable']]
        coords_t = np.array(sdata.obsm['spatial'])

        coords_raw,exps = extract_coords_exp(sdata, batch_key = 'batch', cols = 'spatial', count_layer = '.X', data_format = 'norm1e4')
        print(coords_raw)
    #torch.save(coords_raw, f'/home/data/yulingzhu/mHy/coords_raw.pt')
    
# set the parameters for CAST Mark
        args = Args(
            dataname='task1', # name of the dataset, used to save the log file
            gpu = 0, # gpu id, set to zero for single-GPU nodes
            epochs=400, # number of epochs for training
            lr1= 1e-3, # learning rate
            wd1= 0, # weight decay
            lambd= 1e-3, # lambda in the loss function, refer to online methods
            n_layers=2, # number of GCNII layers, more layers mean a deeper model, larger reception field, at a cost of VRAM usage and computation time
            der=0.5, # edge dropout rate in CCA-SSG
            dfr=0.3, # feature dropout rate in CCA-SSG
            use_encoder=True, # perform a single-layer dimension reduction before the GNNs, helps save VRAM and computation time if the gene panel is large
            encoder_dim=512, # encoder dimension, ignore if `use_encoder` set to `False`
        )
        np.random.seed(j)
        torch.manual_seed(j) 
        if torch.cuda.is_available():
            torch.cuda.manual_seed(j)
            torch.cuda.manual_seed_all(j)
# run CAST Mark: a GNN model equipped with graph convolutional network via initial residual and identity mapping (GCNII) layer
# embed_dict - The dictionary of the graph embedding generated by CAST Mark with sample name. 
        embed_dict = CAST_MARK(coords_raw,exps,fold_path, args = args,graph_strategy='delaunay')
    #print(embed_dict[str(slice_ids[i])].shape)
        #kmeans_plot_multiple(embed_dict,sample_list,coords_raw,'demo1', fold_path, k = 20,dot_size = 10,minibatch = True)
    # set the parameters for CAST Stack
        query_sample = sample_list[0]
        params_dist = reg_params(dataname = query_sample,
                            gpu = 0 if torch.cuda.is_available() else -1,
                            #### Affine parameters
                            iterations=150,
                            dist_penalty1=0,
                            bleeding=500,
                            d_list = [3,2,1,1/2,1/3],
                            attention_params = [None,3,1,0],
                            #### FFD parameters
                            dist_penalty2 = [0],
                            alpha_basis_bs = [0],
                            meshsize = [8],
                            iterations_bs = [1],
                            attention_params_bs = [[None,3,1,0]],
                            mesh_weight = [None])
    # set the alpha basis for the affine transformation
        params_dist.alpha_basis = torch.Tensor([1/1000,1/1000,1/50,5,5]).reshape(5,1).to(params_dist.device)

        coord_final = CAST_STACK(coords_raw,embed_dict, fold_path, sample_list,params_dist,sub_node_idxs = None)
        end_time = time.time()
        elapsed_time = end_time - start_time 
        time_3d.append(elapsed_time)
        dataset = str(slice_ids[i-1]) + '_' + str(slice_ids[i])
        ds.append(dataset)
        #kmeans_plot_multiple(embed_dict,sample_list,coord_final,'demo1_new', fold_path, k=20,dot_size = 30,minibatch=True,plot_strategy='stack')
        slices[i].obsm['spatial'] = coord_final[query_sample].numpy()
        slices_aligned.append(slices[i])
    ##### save coordinates 
    aligned_adata = anndata.concat(slices_aligned)
    aligned_adata.obsm['spatial_3D'] = np.concatenate([aligned_adata.obsm['spatial'],np.array(aligned_adata.obs['Z'].values)[:,None]],axis=1)
    df = pd.DataFrame(aligned_adata.obsm['spatial_3D'],
                columns = ['X', 'Y','Z'],
                index = aligned_adata.obs_names.to_list())
    df['original_clusters'] = aligned_adata.obs['original_clusters'].values
    df['slice'] = aligned_adata.obs['slice_id'].values
    df.to_csv(osp.join(fold_path,'3d_coord_151507.csv')) 
    ##### record time    
    run_times = pd.DataFrame({'3d': time_3d},index = ds)
    run_times.to_csv(osp.join(fold_path,'RunTime_3d.csv'))
    if j==2024:
        run_time_align_iter = time_3d
    else:
        run_time_align_iter = np.add(run_time_align_iter, time_3d)

    run_times_1 = pd.DataFrame({'3d': sum(time_3d)},index = [0])
    run_times_1.to_csv(osp.join(fold_path,'overall_runtime.csv'))
    run_time_iter += sum(time_3d)
    ### save alignment pi 
    for index in range(len(slices)-1):
        f1 = df.loc[df['slice'] == slice_ids[index],['X','Y']]
        f2 = df.loc[df['slice'] == slice_ids[index+1],['X','Y']]
        alignment = euclidean_distances(f1, f2)
        alignment = np.exp(-alignment)
        dataset = str(slice_ids[index]) + '_' + str(slice_ids[index+1])
        map_path = osp.join(fold_path, dataset+'_AlignmentPi.pkl')
        
        with open(map_path,'wb') as f:
            pkl.dump(alignment, f)
run_time_total_iter = run_time_iter/len(seeds_to_run)
run_times_total_iter = pd.DataFrame({'3d':run_time_total_iter},index=[0])
run_times_total_iter.to_csv(osp.join(path_,'overall_runtime.csv'))
# average time for 3d
run_time_align_iter = run_time_align_iter/len(seeds_to_run)
run_times_align_iter = pd.DataFrame({'3d':run_time_align_iter},index=ds)
run_times_align_iter.to_csv(osp.join(path_,'RunTime_3d.csv'))

Simulation data

In [ ]:
slice_ids =[151673,100,80,60,40,20]

def make_unique(s):
    if s in counts:
        counts[s] += 1
    else:
        counts[s] = 0
    return s if counts[s] == 0 else f"{s}_{counts[s]}"  
    # load data and normalize
slices=[]
slice1=sc.read_h5ad('/home/data3/yuling/xenium_region2_5599/simData/DLPFC_151673original_spotIndex.h5')
slice1.layers["counts"] = slice1.X.copy()
#sc.pp.normalize_total(slice1)
#sc.pp.log1p(slice1)
slice1.obs['slice_id']=151673
slice1.obs['Z']=0
slice1.obs_names_make_unique()
slices.append(slice1)
for index,s in enumerate(slice_ids[1:]):
    slice1=sc.read_h5ad(osp.join('/home/data3/yuling/xenium_region2_5599/simData/DLPFC_151673_overlap='+str(s)+'__pseudocount_0_spotIndex.h5'))
    slice1.layers["counts"] = slice1.X.copy()
    #sc.pp.normalize_total(slice1)
    #sc.pp.log1p(slice1)
    slice1.obs['slice_id']=s
    slice1.obs['Z']=index+1
    counts = {}
    slice1.obs_names_make_unique()
    slices.append(slice1) 

In [ ]:
path_ = '/home/data3/yuling/xenium_region2_5599/CAST-main/res/simu_data'
seeds_to_run=[2024,1,2,3,4,10,11,12,13,14]
run_time_iter = 0

for j in seeds_to_run:
    slices_aligned = []
    ds = []
    fold_path = osp.join(path_, str(j))
    if not os.path.exists(fold_path):
        os.makedirs(fold_path)
    slices_aligned.append(slices[0])
    time_3d = []
    for i in range(1,len(slice_ids)):
        start_time = time.time()
        slice1 = slices[i]
        slice2 = slices[i-1]
        sdata = slice1.concatenate(slice2) 

# combine dataset 
        batch_key = 'batch'
        sample_list= [str(slice_ids[i]),str(slice_ids[i-1])] # [Query, Reference]
        batch_rename = {'0' : sample_list[0],'1' : sample_list[1]}
        sdata.obs.replace({batch_key:batch_rename},inplace=True)

# Filter for highly variable genes
        sdata.var['highly_variable'] = detect_highly_variable_genes(sdata,batch_key='batch',n_top_genes=4000,count_layer='.X')
        sdata = sdata[:,sdata.var['highly_variable']]
        coords_t = np.array(sdata.obsm['spatial'])
        coords_raw,exps = extract_coords_exp(sdata, batch_key = 'batch', cols = 'spatial', count_layer = '.X', data_format = 'norm1e4')
    
# set the parameters for CAST Mark
        args = Args(
            dataname='task1', # name of the dataset, used to save the log file
            gpu = 0, # gpu id, set to zero for single-GPU nodes
            epochs=400, # number of epochs for training
            lr1= 1e-3, # learning rate
            wd1= 0, # weight decay
            lambd= 1e-3, # lambda in the loss function, refer to online methods
            n_layers=2, # number of GCNII layers, more layers mean a deeper model, larger reception field, at a cost of VRAM usage and computation time
            der=0.5, # edge dropout rate in CCA-SSG
            dfr=0.3, # feature dropout rate in CCA-SSG
            use_encoder=True, # perform a single-layer dimension reduction before the GNNs, helps save VRAM and computation time if the gene panel is large
            encoder_dim=512, # encoder dimension, ignore if `use_encoder` set to `False`
        )
        np.random.seed(j)
        torch.manual_seed(j) 
        if torch.cuda.is_available():
            torch.cuda.manual_seed(j)
            torch.cuda.manual_seed_all(j) 
# run CAST Mark: a GNN model equipped with graph convolutional network via initial residual and identity mapping (GCNII) layer
# embed_dict - The dictionary of the graph embedding generated by CAST Mark with sample name. 
        embed_dict = CAST_MARK(coords_raw,exps,fold_path, args = args,graph_strategy='delaunay')

        #kmeans_plot_multiple(embed_dict,sample_list,coords_raw,'demo1', fold_path, k=20,dot_size = 10,minibatch=True)
    # set the parameters for CAST Stack
        query_sample = sample_list[0]
        params_dist = reg_params(dataname = query_sample,
                            gpu = 0 if torch.cuda.is_available() else -1,
                            #### Affine parameters
                            iterations=150,
                            dist_penalty1=0,
                            bleeding=500,
                            d_list = [3,2,1,1/2,1/3],
                            attention_params = [None,3,1,0],
                            #### FFD parameters
                            dist_penalty2 = [0],
                            alpha_basis_bs = [0],
                            meshsize = [8],
                            iterations_bs = [1],
                            attention_params_bs = [[None,3,1,0]],
                            mesh_weight = [None])
# set the alpha basis for the affine transformation
        params_dist.alpha_basis = torch.Tensor([1/1000,1/1000,1/50,5,5]).reshape(5,1).to(params_dist.device)

# record time 
        coord_final = CAST_STACK(coords_raw,embed_dict, fold_path, sample_list,params_dist,sub_node_idxs = None)
        end_time = time.time()
        elapsed_time = end_time - start_time 
        time_3d.append(elapsed_time)
        dataset = str(slice_ids[i-1]) + '_' + str(slice_ids[i])
        ds.append(dataset)

        #kmeans_plot_multiple(embed_dict,sample_list,coord_final,'demo1_new', fold_path, k=20,dot_size = 30,minibatch=True,plot_strategy='stack')
        slices[i].obsm['spatial'] = coord_final[query_sample].numpy()
        slices_aligned.append(slices[i])
    ##### save coordinates 
    aligned_adata = anndata.concat(slices_aligned)
    aligned_adata.obsm['spatial_3D']=np.concatenate([aligned_adata.obsm['spatial'],np.array(aligned_adata.obs['Z'].values)[:,None]],axis=1)
    df = pd.DataFrame(aligned_adata.obsm['spatial_3D'],
                columns = ['X', 'Y','Z'],
                index = aligned_adata.obs_names.to_list())
    df['original_clusters'] = aligned_adata.obs['original_clusters'].values
    df['slice'] = aligned_adata.obs['slice_id'].values
    df.to_csv(osp.join(fold_path,'3d_coord_simu.csv')) 
    ##### record time    
    run_times = pd.DataFrame({'3d': time_3d},index = ds)
    run_times.to_csv(osp.join(fold_path,'RunTime_3d.csv'))
    if j==2024:
        run_time_align_iter = time_3d
    else:
        run_time_align_iter = np.add(run_time_align_iter, time_3d)

    run_times_1 = pd.DataFrame({'3d': sum(time_3d)},index = [0])
    run_times_1.to_csv(osp.join(fold_path,'overall_runtime.csv'))
    run_time_iter += sum(time_3d)
    ### save alignment pi 
    for index in range(len(slices)-1):
        f1 = df.loc[df['slice'] == slice_ids[index],['X','Y']]
        f2 = df.loc[df['slice'] == slice_ids[index+1],['X','Y']]
        alignment = euclidean_distances(f1, f2)
        alignment = np.exp(-alignment)
        dataset = str(slice_ids[index]) + '_' + str(slice_ids[index+1])
        map_path = osp.join(fold_path, dataset+'_AlignmentPi.pkl')
        
        with open(map_path,'wb') as f:
            pkl.dump(alignment, f)
    
    for index in range(1, len(slices)):
        f1 = df.loc[df['slice'] == slice_ids[0],['X','Y']]
        f2 = df.loc[df['slice'] == slice_ids[index],['X','Y']]
        alignment = euclidean_distances(f1, f2)
        alignment = np.exp(-alignment)
        dataset = str(slice_ids[0]) + '_' + str(slice_ids[index])
        map_path = osp.join(fold_path, dataset+'_AlignmentPi.pkl')
        
        with open(map_path,'wb') as f:
            pkl.dump(alignment, f)

run_time_total_iter = run_time_iter/len(seeds_to_run)
run_times_total_iter = pd.DataFrame({'3d':run_time_total_iter},index=[0])
run_times_total_iter.to_csv(osp.join(path_,'overall_runtime.csv'))
# average time for 3d
run_time_align_iter = run_time_align_iter/len(seeds_to_run)
run_times_align_iter = pd.DataFrame({'3d':run_time_align_iter},index=ds)
run_times_align_iter.to_csv(osp.join(path_,'RunTime_3d.csv'))


mHypothalamus data

In [ ]:
slices=[]
slice_ids = ['-0.04', '-0.09',  '-0.14', '-0.19' , '-0.24']
for j,section_ids in enumerate(slice_ids):
    slice1 = load_mHypothalamus(section_id=section_ids)
    slice1.var_names_make_unique()
    slice1.layers["counts"] = slice1.X.copy()
    slice1.obs['Z']=j
    slice1.obs['slice_id']=section_ids
    slices.append(slice1)

In [ ]:
path_ = '/home/data3/yuling/xenium_region2_5599/CAST-main/res/mHy'
seeds_to_run = [2024,1,2,3,4,10,11,12,13,14]
run_time_iter = 0

for j in seeds_to_run:
    slices_aligned = []
    ds = []
    fold_path = osp.join(path_, str(j))
    if not os.path.exists(fold_path):
        os.makedirs(fold_path)
    slices_aligned.append(slices[0])
    time_3d = []
    for i in range(1,len(slice_ids)):
        start_time = time.time()
        slice1 = slices[i]
        slice2 = slices[i-1]
        sdata = slice1.concatenate(slice2) 

# combine dataset 
        batch_key = 'batch'
        sample_list= [str(slice_ids[i]),str(slice_ids[i-1])] # [Query, Reference]
        batch_rename = {'0' : sample_list[0],'1' : sample_list[1]}
        sdata.obs.replace({batch_key:batch_rename},inplace=True)

# Filter for highly variable genes
        sdata.var['highly_variable'] = detect_highly_variable_genes(sdata,batch_key='batch',n_top_genes=4000,count_layer='.X')
        sdata = sdata[:,sdata.var['highly_variable']]
        coords_t = np.array(sdata.obsm['spatial'])
        coords_raw,exps = extract_coords_exp(sdata, batch_key = 'batch', cols = 'spatial', count_layer = '.X', data_format = 'norm1e4')
    
# set the parameters for CAST Mark
        args = Args(
            dataname='task1', # name of the dataset, used to save the log file
            gpu = 0, # gpu id, set to zero for single-GPU nodes
            epochs=400, # number of epochs for training
            lr1= 1e-3, # learning rate
            wd1= 0, # weight decay
            lambd= 1e-3, # lambda in the loss function, refer to online methods
            n_layers=2, # number of GCNII layers, more layers mean a deeper model, larger reception field, at a cost of VRAM usage and computation time
            der=0.5, # edge dropout rate in CCA-SSG
            dfr=0.3, # feature dropout rate in CCA-SSG
            use_encoder=True, # perform a single-layer dimension reduction before the GNNs, helps save VRAM and computation time if the gene panel is large
            encoder_dim=512, # encoder dimension, ignore if `use_encoder` set to `False`
        )
        np.random.seed(j)
        torch.manual_seed(j) 
        if torch.cuda.is_available():
            torch.cuda.manual_seed(j)
            torch.cuda.manual_seed_all(j) 
# run CAST Mark: a GNN model equipped with graph convolutional network via initial residual and identity mapping (GCNII) layer
# embed_dict - The dictionary of the graph embedding generated by CAST Mark with sample name. 
        embed_dict = CAST_MARK(coords_raw,exps,fold_path, args = args,graph_strategy='delaunay')

        #kmeans_plot_multiple(embed_dict,sample_list,coords_raw,'demo1', fold_path, k=20,dot_size = 10,minibatch=True)
    # set the parameters for CAST Stack
        query_sample = sample_list[0]
        params_dist = reg_params(dataname = query_sample,
                            gpu = 0 if torch.cuda.is_available() else -1,
                            #### Affine parameters
                            iterations=150,
                            dist_penalty1=0,
                            bleeding=500,
                            d_list = [3,2,1,1/2,1/3],
                            attention_params = [None,3,1,0],
                            #### FFD parameters
                            dist_penalty2 = [0],
                            alpha_basis_bs = [0],
                            meshsize = [8],
                            iterations_bs = [1],
                            attention_params_bs = [[None,3,1,0]],
                            mesh_weight = [None])
# set the alpha basis for the affine transformation
        params_dist.alpha_basis = torch.Tensor([1/1000,1/1000,1/50,5,5]).reshape(5,1).to(params_dist.device)

# record time 
        coord_final = CAST_STACK(coords_raw,embed_dict, fold_path, sample_list,params_dist,sub_node_idxs = None)
        end_time = time.time()
        elapsed_time = end_time - start_time 
        time_3d.append(elapsed_time)
        dataset = str(slice_ids[i-1]) + '_' + str(slice_ids[i])
        ds.append(dataset)

        #kmeans_plot_multiple(embed_dict,sample_list,coord_final,'demo1_new', fold_path, k=20,dot_size = 30,minibatch=True,plot_strategy='stack')
        slices[i].obsm['spatial'] = coord_final[query_sample].numpy()
        slices_aligned.append(slices[i])
    ##### save coordinates 
    aligned_adata = anndata.concat(slices_aligned)
    aligned_adata.obsm['spatial_3D'] = np.concatenate([aligned_adata.obsm['spatial'],np.array(aligned_adata.obs['Z'].values)[:,None]],axis=1)
    df = pd.DataFrame(aligned_adata.obsm['spatial_3D'],
                columns = ['X', 'Y','Z'],
                index = aligned_adata.obs_names.to_list())
    df['original_clusters'] = aligned_adata.obs['original_clusters'].values
    df['slice'] = aligned_adata.obs['slice_id'].values
    df.to_csv(osp.join(fold_path,'3d_coord_mHy.csv')) 
    ##### record time    
    run_times = pd.DataFrame({'3d': time_3d},index = ds)
    run_times.to_csv(osp.join(fold_path,'RunTime_3d.csv'))
    if j == 2024:
        run_time_align_iter = time_3d
    else:
        run_time_align_iter = np.add(run_time_align_iter, time_3d)

    run_times_1 = pd.DataFrame({'3d': sum(time_3d)},index = [0])
    run_times_1.to_csv(osp.join(fold_path,'overall_runtime.csv'))
    run_time_iter += sum(time_3d)
    ### save alignment pi 
    for index in range(len(slices)-1):
        f1 = df.loc[df['slice'] == slice_ids[index],['X','Y']]
        f2 = df.loc[df['slice'] == slice_ids[index+1],['X','Y']]
        alignment = euclidean_distances(f1, f2)
        alignment = np.exp(-alignment)
        dataset = str(slice_ids[index]) + '_' + str(slice_ids[index+1])
        map_path = osp.join(fold_path, dataset+'_AlignmentPi.pkl')
        
        with open(map_path,'wb') as f:
            pkl.dump(alignment, f)

run_time_total_iter = run_time_iter/len(seeds_to_run)
run_times_total_iter = pd.DataFrame({'3d':run_time_total_iter},index = [0])
run_times_total_iter.to_csv(osp.join(path_,'overall_runtime.csv'))
# average time for 3d
run_time_align_iter = run_time_align_iter/len(seeds_to_run)
run_times_align_iter = pd.DataFrame({'3d':run_time_align_iter},index=ds)
run_times_align_iter.to_csv(osp.join(path_,'RunTime_3d.csv'))


Merfish data

In [ ]:

in_ = '/home/data3/yuling/xenium_region2_5599/merfish_mouse_brain'
slice_ids_o = range(33)
sliceID =[[str(x) for x in slice_ids_o]]
end_no = 33
slices = [] #this is the original data
slice_ids = sliceID[0]
for i in range(len(slice_ids)):
    slice1 = sc.read(osp.join(in_, 'merfish_mouse_brain_slice'+slice_ids[i]+'.h5ad'))
    slice1.obs['original_clusters']=slice1.obs['spa_cluster'].astype('category')
    slice1.obsm['spatial'] = slice1.obsm['spatial'].to_numpy()[:, :2]
    slice1.obs['Z']=i
    slice1.obs['slice_id'] = slice_ids[i]
    slices.append(slice1)

In [ ]:
path_ = '/home/data3/yuling/xenium_region2_5599/CAST-main/res/Merfish_mouse'
seeds_to_run = [2024,1,2,3,4,10,11,12,13,14]
run_time_iter = 0

for j in seeds_to_run:
    slices_aligned = []
    ds = []
    fold_path = osp.join(path_, str(j))
    if not os.path.exists(fold_path):
        os.makedirs(fold_path)
    slices_aligned.append(slices[0])
    time_3d = []
    for i in range(1,len(slice_ids)):
        start_time = time.time()
        slice1 = slices[i]
        slice2 = slices[i-1]
        sdata = slice1.concatenate(slice2) 

# combine dataset 
        batch_key = 'batch'
        sample_list= [str(slice_ids[i]),str(slice_ids[i-1])] # [Query, Reference]
        batch_rename = {'0' : sample_list[0],'1' : sample_list[1]}
        sdata.obs.replace({batch_key:batch_rename},inplace=True)

# Filter for highly variable genes
        sdata.var['highly_variable'] = detect_highly_variable_genes(sdata,batch_key='batch',n_top_genes=4000,count_layer='.X')
        sdata = sdata[:,sdata.var['highly_variable']]
        coords_t = np.array(sdata.obsm['spatial'])

        coords_raw,exps = extract_coords_exp(sdata, batch_key = 'batch', cols = 'spatial', count_layer = '.X', data_format = 'norm1e4')
    
# set the parameters for CAST Mark
        args = Args(
            dataname='task1', # name of the dataset, used to save the log file
            gpu = 0, # gpu id, set to zero for single-GPU nodes
            epochs=400, # number of epochs for training
            lr1= 1e-3, # learning rate
            wd1= 0, # weight decay
            lambd= 1e-3, # lambda in the loss function, refer to online methods
            n_layers=2, # number of GCNII layers, more layers mean a deeper model, larger reception field, at a cost of VRAM usage and computation time
            der=0.5, # edge dropout rate in CCA-SSG
            dfr=0.3, # feature dropout rate in CCA-SSG
            use_encoder=True, # perform a single-layer dimension reduction before the GNNs, helps save VRAM and computation time if the gene panel is large
            encoder_dim=512, # encoder dimension, ignore if `use_encoder` set to `False`
        )
        np.random.seed(j)
        torch.manual_seed(j) 
        if torch.cuda.is_available():
            torch.cuda.manual_seed(j)
            torch.cuda.manual_seed_all(j)
# run CAST Mark: a GNN model equipped with graph convolutional network via initial residual and identity mapping (GCNII) layer
# embed_dict - The dictionary of the graph embedding generated by CAST Mark with sample name. 
        embed_dict = CAST_MARK(coords_raw,exps,fold_path, args = args,graph_strategy='delaunay')
    
        #kmeans_plot_multiple(embed_dict,sample_list,coords_raw,'demo1', fold_path, k=20,dot_size = 10,minibatch=True)
    # set the parameters for CAST Stack
        query_sample = sample_list[0]
        params_dist = reg_params(dataname = query_sample,
                            gpu = 0 if torch.cuda.is_available() else -1,
                            #### Affine parameters
                            iterations = 150,
                            dist_penalty1=0,
                            bleeding = 1000,
                            d_list = [3,2,1,1/2,1/3],
                            attention_params = [None,3,1,0],
                            #### FFD parameters
                            dist_penalty2 = [0],
                            alpha_basis_bs = [0],
                            meshsize = [8],
                            iterations_bs = [1],
                            attention_params_bs = [[None,3,1,0]],
                            mesh_weight = [None])

# set the alpha basis for the affine transformation
        params_dist.alpha_basis = torch.Tensor([1/1000,1/1000,1/50,5,5]).reshape(5,1).to(params_dist.device)

        coord_final = CAST_STACK(coords_raw,embed_dict, fold_path, sample_list,params_dist,sub_node_idxs = None)
        end_time = time.time()
        elapsed_time = end_time - start_time 
        time_3d.append(elapsed_time)
        dataset = str(slice_ids[i-1]) + '_' + str(slice_ids[i])
        ds.append(dataset)
        #kmeans_plot_multiple(embed_dict,sample_list,coord_final,'demo1_new', fold_path, k=20,dot_size = 30,minibatch=True,plot_strategy='stack')
        slices[i].obsm['spatial'] = coord_final[query_sample].numpy()
        slices_aligned.append(slices[i])
    ##### save coordinates 
    aligned_adata = anndata.concat(slices_aligned)
    aligned_adata.obsm['spatial_3D'] = np.concatenate([aligned_adata.obsm['spatial'],np.array(aligned_adata.obs['Z'].values)[:,None]],axis=1)
    df = pd.DataFrame(aligned_adata.obsm['spatial_3D'],
                columns = ['X', 'Y','Z'],
                index = aligned_adata.obs_names.to_list())
    df['original_clusters'] = aligned_adata.obs['original_clusters'].values
    df['slice'] = aligned_adata.obs['slice_id'].values
    df.to_csv(osp.join(fold_path,'3d_coord_merfish.csv')) 
    ##### record time    
    run_times = pd.DataFrame({'3d': time_3d},index = ds)
    run_times.to_csv(osp.join(fold_path,'RunTime_3d.csv'))
    if j==2024:
        run_time_align_iter = time_3d
    else:
        run_time_align_iter = np.add(run_time_align_iter, time_3d)

    run_times_1 = pd.DataFrame({'3d': sum(time_3d)},index = [0])
    run_times_1.to_csv(osp.join(fold_path,'overall_runtime.csv'))
    run_time_iter += sum(time_3d)
    ### save alignment pi 
    for index in range(len(slices)-1):
        f1 = df.loc[df['slice'] == slice_ids[index],['X','Y']]
        f2 = df.loc[df['slice'] == slice_ids[index+1],['X','Y']]
        alignment = euclidean_distances(f1, f2)
        alignment = np.exp(-alignment)
        dataset = str(slice_ids[index]) + '_' + str(slice_ids[index+1])
        map_path = osp.join(fold_path, dataset+'_AlignmentPi.pkl')
        
        with open(map_path,'wb') as f:
            pkl.dump(alignment, f)
run_time_total_iter = run_time_iter/len(seeds_to_run)
run_times_total_iter = pd.DataFrame({'3d':run_time_total_iter},index=[0])
run_times_total_iter.to_csv(osp.join(path_,'overall_runtime.csv'))
# average time for 3d
run_time_align_iter = run_time_align_iter/len(seeds_to_run)
run_times_align_iter = pd.DataFrame({'3d':run_time_align_iter},index=ds)
run_times_align_iter.to_csv(osp.join(path_,'RunTime_3d.csv'))